# Construcción del recuperador y contrato E5
Construye BM25 sobre el corpus público. La ruta E5 usa un endpoint explícito; su ausencia no se reemplaza por vectores ficticios.


In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
assert (root / "src").exists(), "Ejecute desde la raíz o desde notebooks"
sys.path.insert(0, str(root))


In [2]:
from src.utils import read_jsonl
from src.model import BM25
from src.train import build
index = BM25(read_jsonl(root / "data/processed/chunks.jsonl"))
build()

{'method': 'BM25',
 'k1': 1.5,
 'b': 0.75,
 'top_k': 5,
 'corpus_sha256': '7ebe3bf6a4f97c1b8ebba4dd5bde3845220e28ce6f619dab413327584756e9a0',
 'chunks': 271,
 'fit_llm': False}

In [3]:
questions = ["¿Qué actividades excluye el GLF?", "¿Qué es el PGAS?"]
for question in questions:
    print(question)
    print([{k: r[k] for k in ["chunk_id", "source_name", "page", "score"]} for r in index.search(question)])

¿Qué actividades excluye el GLF?
[{'chunk_id': 'GLF_MANUAL_SGAS_PUBLIC::P24C01', 'source_name': 'Manual SGAS', 'page': 24, 'score': 6.318623}, {'chunk_id': 'GLF_MANUAL_SGAS_PUBLIC::P24C02', 'source_name': 'Manual SGAS', 'page': 24, 'score': 5.90262}, {'chunk_id': 'GLF_MANUAL_SGAS_PUBLIC::P11C04', 'source_name': 'Manual SGAS', 'page': 11, 'score': 2.841931}, {'chunk_id': 'GLF_ANEXO_B_PUBLIC::P02C01', 'source_name': 'Anexo B - Lista de Exclusión', 'page': 2, 'score': 2.568649}, {'chunk_id': 'GLF_ANEXO_A_PUBLIC::P04C01', 'source_name': 'Anexo A - Política Ambiental y Social', 'page': 4, 'score': 2.407354}]
¿Qué es el PGAS?
[{'chunk_id': 'GLF_MANUAL_SGAS_PUBLIC::P25C03', 'source_name': 'Manual SGAS', 'page': 25, 'score': 3.628996}, {'chunk_id': 'GLF_MANUAL_SGAS_PUBLIC::P22C02', 'source_name': 'Manual SGAS', 'page': 22, 'score': 3.550999}, {'chunk_id': 'GLF_MANUAL_SGAS_PUBLIC::P23C03', 'source_name': 'Manual SGAS', 'page': 23, 'score': 3.550999}, {'chunk_id': 'GLF_ANEXO_H_PUBLIC::P03C03', '

In [4]:
import os
from src.model import E5Endpoint
endpoint = os.getenv("GLF_E5_ENDPOINT")
if endpoint:
    semantic = E5Endpoint(index.records, endpoint, os.getenv("GLF_E5_API_KEY", ""))
    print({"e5_model": semantic.name, "dimensions": 384, "same_corpus": [r["chunk_id"] for r in semantic.records] == [r["chunk_id"] for r in index.records]})
else:
    print("E5 no ejecutado: endpoint no configurado. Las pruebas de contrato no son un benchmark del modelo.")

E5 no ejecutado: endpoint no configurado. Las pruebas de contrato no son un benchmark del modelo.
